In [1]:
# as bibliotecas que irei usar
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader, TensorDataset

In [2]:
# define o dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [3]:
# dados de entrada
x_train = torch.tensor([
    [[[4,5,6,7],[5,6,7,8],[8,9,10,11],[4,6,7,8]]],
    [[[-4,5,6,-7],[5,-6,7,8],[-8,9,-10,11],[-4,-6,-7,-8]]]
]).float().to(device)

In [4]:
# normaliza os pixels
x_train.div_(8)

# rotulos
y_train = torch.tensor([0, 1]).float().to(device)

print('Shape x_train:', x_train.shape)
print('Shape y_train:', y_train.shape)

Shape x_train: torch.Size([2, 1, 4, 4])
Shape y_train: torch.Size([2])


In [5]:
# arquitetura da CNN
def get_model():
    model = nn.Sequential(
        nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3),
        nn.MaxPool2d(kernel_size=2), # reduz a resolução
        nn.ReLU(), # zera valores negativos
        nn.Flatten(), # achata para vetor 1D
        nn.Linear(1, 1), # camada de decisão
        nn.Sigmoid() # converte para probabilidade
    ).to(device)

    loss_fn   = nn.BCELoss() # perda para classificação
    optimizer = Adam(model.parameters(), lr=0.01) # otimizador
    return model, loss_fn, optimizer

model, criterion, optimizer = get_model()
print(model)

Sequential(
  (0): Conv2d(1, 1, kernel_size=(3, 3), stride=(1, 1))
  (1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (2): ReLU()
  (3): Flatten(start_dim=1, end_dim=-1)
  (4): Linear(in_features=1, out_features=1, bias=True)
  (5): Sigmoid()
)


In [6]:
# treina um batch
def train_batch(x, y, model, optimizer, loss_fn):
    model.train() # modo de treino
    optimizer.zero_grad() # zera os gradientes
    predictions = model(x) # forward pass
    batch_loss  = loss_fn(predictions.squeeze(), y.squeeze()) # calcula o erro
    batch_loss.backward() # backward pass
    optimizer.step() # atualiza os pesos
    return batch_loss.item()

In [7]:
# agrupa x e y em um TensorDataset
trn_dl = DataLoader(TensorDataset(x_train, y_train))

In [9]:
# loop de treinamento por 2000 épocas
for epoch in range(2000):
    for ix, batch in enumerate(trn_dl):
        x, y = batch
        x, y = x.to(device), y.to(device)
        loss = train_batch(x, y, model, optimizer, criterion)

    # exibe o progresso a cada 500 épocas
    if epoch % 500 == 0:
        print(f'Época {epoch:4d} Loss: {loss:.4f}')

Época    0 Loss: 0.0031
Época  500 Loss: 0.0017
Época 1000 Loss: 0.0010
Época 1500 Loss: 0.0006


In [11]:
# verifica a predição final
model.eval()
with torch.no_grad():
    preds = model(x_train).squeeze()
    for i, (pred, label) in enumerate(zip(preds, y_train)):
        print(f'Imagem {i} Predição: {pred:.4f} Rótulo real: {int(label)}')

Imagem 0 Predição: 0.0000 Rótulo real: 0
Imagem 1 Predição: 0.9997 Rótulo real: 1
